# Analysis COSI Dataset: Label Preciction

Note that this is just a notebook for an exaple since we run the entire data set on GPU to be faster


**Data Description**:
- id user
- sex
- age
- code shop --> location shop in French department
- survey answer Q1
- survey answer Q2
- survey answer Q3
- HA: true/false

In [ ]:
import numpy as np
import os as os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sn
from matplotlib import cm as CM
import random
import pandas as pd
color = sn.color_palette()
%matplotlib inline

import plotly.offline as py
import plotly.graph_objs as go
import plotly.tools as tls
import geopandas as gpd
from shapely.geometry import Point
from geopy.geocoders import Nominatim
import time
import missingno as msno
from matplotlib.colors import ListedColormap
from matplotlib.colors import LinearSegmentedColormap
from summarizer import Summarizer
import nltk
from nltk.corpus import stopwords
from wordcloud import WordCloud

In [ ]:
from transformers import CamembertTokenizer, CamembertModel
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

In [ ]:
import torch
from tqdm import tqdm, trange

In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from tensorflow.keras.preprocessing.sequence import pad_sequences
from transformers import CamembertTokenizer, CamembertForSequenceClassification

In [ ]:
from transformers import AdamW

In [ ]:
# Defining constants
epochs = 5
MAX_LEN = 128
batch_size = 16
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
label_mapping = {'Autres': 0, 
                 'Dépistage': 1, 
                 'Conversation à 1 ou 2 dans le bruit': 2,
                 'Conversation en groupe dans le silence': 3, 
                 'Conversation à 1 ou 2 dans le silence': 4,
                 'Conversation à 1 ou 2 dans le bruit': 5,
                 'Television/radio au volume normal': 6, 
                 'Conversation en groupe dans le bruit': 7, 
                 'Augmenter le contact social': 8,
                 'Eglise ou reunion': 9, 
                 'Se sentir embarrasse ou stupide': 10,
                'Interlocuteur non familier au telephone': 11,
                'Interlocuteur familier au telephone': 12,
                 "Entendre la sonnette de la porte ou quelqu'un frapper": 13, 
                 'Entendre le trafic': 14,
                "Entendre le telephone sonner d'une autre piece": 15,
                'Se sentir exclu': 16}  # Define a mapping from string labels to numerical values

In [ ]:
# Initialize CamemBERT tokenizer
tokenizer = CamembertTokenizer.from_pretrained('camembert-base',do_lower_case=True)

In [ ]:
model = torch.load('full_model.pth')

# Test the model on an unseen texts - Q1

In [ ]:
##############
#READING DATA#
##############
# COSI Q1 
df_test_Q1 = pd.read_excel('./COSI_LABELS_test.xlsx', sheet_name=0)

In [ ]:
msno.matrix(df_test_Q1)

In [ ]:
missing_percent_q1 = df_test_Q1.isna().mean() * 100  # Returns a Series with percentages of missing values
missing_percent_q1

In [ ]:
comments = df_test_Q1['OPEN_ANSWER']

In [ ]:
len(comments)

In [ ]:
# Encode the comments
tokenized_comments_ids = [tokenizer.encode(comment,add_special_tokens=True,max_length=MAX_LEN) for comment in comments]
# Pad the resulted encoded comments
tokenized_comments_ids = pad_sequences(tokenized_comments_ids, maxlen=MAX_LEN, dtype="long", truncating="post", padding="post")

# Create attention masks 
attention_masks = []
for seq in tokenized_comments_ids:
  seq_mask = [float(i>0) for i in seq]
  attention_masks.append(seq_mask)

prediction_inputs = torch.tensor(tokenized_comments_ids)
prediction_masks = torch.tensor(attention_masks)

In [ ]:
#for i in range(len(flat_pred)):
#    print('Comment: ', comments.iloc[i]) #print('Comment: ', comments[i]) only for today
#    print('Label', flat_pred[i])

In [ ]:
batch_size = 8  # You can adjust this based on your available memory

# Create DataLoader for batching
data = torch.utils.data.TensorDataset(torch.tensor(tokenized_comments_ids), torch.tensor(attention_masks))
dataloader = torch.utils.data.DataLoader(data, batch_size=batch_size)

flat_pred = []

# Apply the finetuned model (Camembert) on batches
with torch.no_grad():
    model.eval()
    for batch in dataloader:
        batch = tuple(t.to(device) for t in batch)
        input_ids, attention_mask = batch

        # Forward pass, calculate logit predictions for each batch
        batch_outputs = model(input_ids, token_type_ids=None, attention_mask=attention_mask)
        batch_logits = batch_outputs[0]
        batch_logits = batch_logits.detach().cpu().numpy()

        flat_pred.extend(np.argmax(batch_logits, axis=1).flatten())

In [ ]:
len(flat_pred)

In [ ]:
for i in range(len(flat_pred)):
    print('Comment: ', comments[i])
    print('Label', flat_pred[i])

In [ ]:
# Create a list of labels based on flat_pred
labels = [list(label_mapping.keys())[list(label_mapping.values()).index(pred)] for pred in flat_pred]

# Create a DataFrame
result_df = pd.DataFrame({'Comments': comments, 'Flat_Pred': flat_pred, 'Labels': labels})


In [ ]:
# Specify the file path where you want to save the Excel file
excel_file_path = './result_Q1.xlsx'

# Export the DataFrame to Excel
result_df.to_excel(excel_file_path, index=False)